# 04. DeepSeek-Coder без обучения

Используем deepseek-coder-1.3b-base. Для каждого кода сравниваем сумму log probability всех токенов двух ответов. Prompt и усечение совпадают с последующим LoRA. Нужны интернет для весов и CUDA для 4 бит.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Open the notebook from the CodeGuard repository"

def run_script(name, *args):
    command = [sys.executable, "-u", str(ROOT / "src" / name), *map(str, args)]
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True,
                          encoding="utf-8", errors="replace",
                          env={**os.environ, "PYTHONIOENCODING": "utf-8"}) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f"{name} failed (exit {returncode}). See the output above.")


Перед запуском проверяем PyTorch и CUDA. `USE_4BIT=True` предназначен для GPU (например Colab T4). Для CPU явно установите `USE_4BIT=False`: веса будут загружены без сжатия, потребуется больше RAM, а оценка будет медленной. Для первой проверки добавьте `"--limit-pairs", 8, "--batch-size", 1` в вызов run_script и выберите отдельный output-каталог. Результаты CPU без сжатия нельзя напрямую приписывать 4-битному baseline.

In [ ]:
import importlib.util
import torch

USE_4BIT = True
print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if USE_4BIT and not torch.cuda.is_available():
    raise RuntimeError(
        "4-bit mode requires CUDA-enabled PyTorch and an available NVIDIA GPU. "
        "Use a Colab GPU runtime, or set USE_4BIT=False to run without quantization on CPU."
    )
if USE_4BIT and importlib.util.find_spec("bitsandbytes") is None:
    raise RuntimeError("Install bitsandbytes in this notebook's Python environment before using 4-bit mode.")

options = ["--quantized"] if USE_4BIT else []
run_script("llm_baseline.py", *options, "--data-dir", "data/processed/small", "--output", "runs/deepseek_zero_shot")


In [ ]:
import json
metrics = json.loads((ROOT / "runs/deepseek_zero_shot/metrics.json").read_text(encoding="utf-8"))
metrics

Сначала можно добавить `--limit-pairs 8 --batch-size 1`. Для оценки без сжатия уберите `--quantized`. Одинаковый режим квантования нужен для честного сравнения с QLoRA. `p_buggy` — относительная вероятность среди двух ответов, не калиброванная вероятность бага.